# 03 — Evaluating the contract analyzer

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — Legal Tech |
| **Level** | Advanced |
| **Time** | ~30 min |
| **Prerequisites** | `02_build.ipynb` |
| **Open in Colab** | `labs/03_contract_analyzer/03_evaluate.ipynb` |

> **Goal:** Rigorously evaluate every component of the contract analyzer —
> segmentation, classification, risk scoring, comparison quality, and
> end-to-end cost-effectiveness.

## Setup

In [ ]:
# ── Install dependencies ──
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0" "scikit-learn>=1.2.0"

import os, json, re, warnings
from typing import List, Dict, Tuple, Any, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    cohen_kappa_score, mean_absolute_error,
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score,
)
warnings.filterwarnings("ignore")

from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

if client:
    print("✓ OpenAI client ready")
else:
    print("⚠  OPENAI_API_KEY not set — LLM-as-judge cells will use mock scores")

print("✓ Dependencies installed")

## 1 · Evaluation framework

We evaluate the contract analyzer across **five dimensions**, each targeting
a specific pipeline component:

| Component | Metric | What it measures |
|---|---|---|
| Segmentation | Precision, Recall, F1 | Are clause boundaries correct? |
| Classification | Accuracy, per-type F1 | Are clause types correctly identified? |
| Risk scoring | Cohen's κ, MAE, confusion matrix | Does AI scoring agree with expert labels? |
| Comparison quality | LLM-as-judge (1–5) | Are standard-clause comparisons relevant and accurate? |
| Negotiation quality | LLM-as-judge (1–5) | Are suggestions legally sound, actionable, and specific? |
| End-to-end | Cost, time, failure modes | Is it practical for production use? |

Each evaluation uses **labeled ground truth** — either expert annotations
or LLM-as-judge scoring with clear rubrics.

In [ ]:
# ── Evaluation data structures ──
from dataclasses import dataclass

@dataclass
class EvalResult:
    """Result of a single evaluation metric."""
    metric_name: str
    value: float
    interpretation: str
    component: str

def summarize_results(results: List[EvalResult]) -> pd.DataFrame:
    """Create a summary table from evaluation results."""
    rows = [{"Component": r.component, "Metric": r.metric_name,
             "Value": f"{r.value:.3f}", "Interpretation": r.interpretation}
            for r in results]
    return pd.DataFrame(rows)

all_results: List[EvalResult] = []
print("✓ Evaluation framework ready")
print("  Collecting results into all_results[] for final summary")

## 2 · Clause segmentation evaluation

We define **ground-truth clause boundaries** for a test contract and measure
how well our segmenter identifies them.

- **Precision**: Of predicted boundaries, how many are correct?
- **Recall**: Of actual boundaries, how many did we find?
- **F1**: Harmonic mean of precision and recall

In [ ]:
# ── Ground-truth segmentation for test contract ──
TEST_CONTRACT_SEGMENTATION = """
1. DEFINITIONS
This Agreement is between AlphaCorp ("Client") and BetaServices ("Provider").

1.1 "Services" means consulting services described in each Statement of Work.

1.2 "Deliverables" means tangible work product specified in each SOW.

2. SCOPE OF SERVICES
Provider shall perform Services as described in executed SOWs.
Provider shall assign qualified personnel approved by Client.

3. COMPENSATION
Client shall pay fees per SOW. Standard terms: net-30 from invoice date.
Late payments accrue interest at 1.5% per month.

4. INTELLECTUAL PROPERTY
Provider retains pre-existing IP. Deliverables created under SOWs
are assigned to Client upon full payment. Provider retains a license
to general methodologies and know-how.

5. CONFIDENTIALITY
Each party protects the other's confidential information for 3 years.
Standard exceptions: public domain, independent development, legal compulsion.

6. LIMITATION OF LIABILITY
Neither party's liability exceeds fees paid in the prior 12 months.
Exclusion of consequential damages for both parties.

7. TERMINATION
Either party may terminate with 60 days notice or immediately for
uncured material breach. Survival clauses: Sections 4, 5, 6.
"""

# Ground truth: each clause boundary as (start_section, clause_count)
GROUND_TRUTH_BOUNDARIES = [
    "1. DEFINITIONS",
    "1.1 Services",
    "1.2 Deliverables",
    "2. SCOPE OF SERVICES",
    "3. COMPENSATION",
    "4. INTELLECTUAL PROPERTY",
    "5. CONFIDENTIALITY",
    "6. LIMITATION OF LIABILITY",
    "7. TERMINATION",
]

# ── Our segmenter ──
def segment_contract(text: str) -> List[Dict]:
    """Segment contract into clauses (re-implementation for eval notebook)."""
    text = re.sub(r'\r\n', '\n', text.strip())
    section_re = re.compile(
        r'\n\s*(?=\d+\.\d*\s+[A-Z]|Section\s+\d+|ARTICLE\s+[IVXLCDM]+)',
        re.MULTILINE
    )
    raw_segments = section_re.split(text)
    clauses = []
    idx = 0
    for segment in raw_segments:
        paragraphs = re.split(r'\n\s*\n', segment.strip())
        for para in paragraphs:
            para = para.strip()
            if not para or len(para) < 25:
                if clauses and para:
                    clauses[-1]["text"] += " " + para
                continue
            hdr_match = re.match(r'^(\d+\.\d*\s+[A-Z][A-Z ]+|[A-Z][A-Z ]{4,})', para)
            header = hdr_match.group(0).strip() if hdr_match else None
            clauses.append({"text": para, "index": idx, "header": header})
            idx += 1
    return clauses

predicted_clauses = segment_contract(TEST_CONTRACT_SEGMENTATION)
n_predicted = len(predicted_clauses)
n_ground_truth = len(GROUND_TRUTH_BOUNDARIES)

# ── Boundary matching (fuzzy) ──
matched = 0
for gt_boundary in GROUND_TRUTH_BOUNDARIES:
    gt_key = gt_boundary.split()[0].lower()  # e.g., "1.", "1.1", "2."
    for pred in predicted_clauses:
        if gt_key in pred["text"][:30].lower():
            matched += 1
            break

precision = matched / n_predicted if n_predicted > 0 else 0
recall = matched / n_ground_truth if n_ground_truth > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Segmentation Evaluation")
print(f"{'─'*50}")
print(f"  Ground truth clauses: {n_ground_truth}")
print(f"  Predicted clauses:    {n_predicted}")
print(f"  Matched:              {matched}")
print(f"")
print(f"  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  F1 Score:  {f1:.3f}")

all_results.append(EvalResult("Precision", precision, f"{'Good' if precision > 0.8 else 'Needs improvement'}", "Segmentation"))
all_results.append(EvalResult("Recall", recall, f"{'Good' if recall > 0.8 else 'Needs improvement'}", "Segmentation"))
all_results.append(EvalResult("F1", f1, f"{'Strong' if f1 > 0.8 else 'Moderate' if f1 > 0.6 else 'Weak'}", "Segmentation"))

print(f"\nPredicted clauses:")
for c in predicted_clauses:
    hdr = f" [{c['header']}]" if c.get('header') else ""
    print(f"  [{c['index']:2d}]{hdr} {c['text'][:70].replace(chr(10), ' ')}…")

## 3 · Risk scoring accuracy

We compare our system's risk scores against **expert-labeled** ground truth
for 24 clauses.  Metrics:

- **Cohen's κ** — inter-rater agreement (accounts for chance)
- **MAE** — mean absolute error between predicted and expert scores
- **Confusion matrix** — where does the system over/under-estimate risk?

In [ ]:
# ── Expert-labeled risk scores (ground truth) ──
EXPERT_LABELED_CLAUSES: List[Dict[str, Any]] = [
    {"text": "Each party indemnifies the other for direct damages, capped at 12 months fees", "expert": 2, "type": "indemnification"},
    {"text": "Vendor indemnifies Client without any cap or limitation whatsoever", "expert": 5, "type": "indemnification"},
    {"text": "Mutual indemnification for third-party IP claims with reasonable cap", "expert": 2, "type": "indemnification"},
    {"text": "Provider assumes all financial liability for any claim of any kind", "expert": 5, "type": "indemnification"},
    {"text": "Neither party liable for more than 12 months of fees paid", "expert": 1, "type": "limitation_of_liability"},
    {"text": "Client liability capped at $100; Vendor liability is unlimited", "expert": 5, "type": "limitation_of_liability"},
    {"text": "Standard consequential damages exclusion for both parties", "expert": 1, "type": "limitation_of_liability"},
    {"text": "Provider waives all limitation of liability defenses", "expert": 5, "type": "limitation_of_liability"},
    {"text": "Vendor retains pre-existing IP; deliverables assigned on payment", "expert": 1, "type": "ip_assignment"},
    {"text": "All IP including pre-existing tools becomes Client property forever", "expert": 5, "type": "ip_assignment"},
    {"text": "Work-for-hire deliverables owned by Client; Vendor keeps methodologies", "expert": 2, "type": "ip_assignment"},
    {"text": "Either party may terminate with 60 days written notice", "expert": 1, "type": "termination"},
    {"text": "Client may terminate at will; Provider cannot terminate ever", "expert": 5, "type": "termination"},
    {"text": "Termination for cause with 30 day cure period", "expert": 1, "type": "termination"},
    {"text": "Automatic termination if Provider stock price drops 10%", "expert": 4, "type": "termination"},
    {"text": "Mutual confidentiality for 3 years with standard exceptions", "expert": 1, "type": "confidentiality"},
    {"text": "Provider keeps all info secret forever, no exceptions, not mutual", "expert": 5, "type": "confidentiality"},
    {"text": "Non-solicitation of employees for 12 months post-termination", "expert": 2, "type": "non_compete"},
    {"text": "Non-compete across all industries worldwide for 5 years", "expert": 5, "type": "non_compete"},
    {"text": "Payment net-30 from invoice with 1% monthly late interest", "expert": 1, "type": "payment_terms"},
    {"text": "Payment net-120, Client may withhold at sole discretion, no interest", "expert": 4, "type": "payment_terms"},
    {"text": "Software warranted for 12 months; sole remedy is re-performance", "expert": 1, "type": "warranty"},
    {"text": "Provider warrants zero defects forever with unlimited liability", "expert": 5, "type": "warranty"},
    {"text": "Binding arbitration under AAA rules in neutral venue", "expert": 1, "type": "dispute_resolution"},
]

# ── Simulate system predictions (heuristic-based for reproducibility) ──
def predict_risk_score(clause_text: str) -> int:
    """Heuristic risk scorer for evaluation purposes."""
    text_lower = clause_text.lower()
    score = 2.0

    risk_signals = [
        ("unlimited" in text_lower or "without any cap" in text_lower, 1.5),
        ("any and all" in text_lower or "any kind" in text_lower, 1.0),
        ("perpetuity" in text_lower or "forever" in text_lower, 1.2),
        ("sole and exclusive" in text_lower, 0.8),
        ("$100" in text_lower, 1.2),
        ("waives" in text_lower, 1.0),
        ("cannot terminate" in text_lower or "may not terminate" in text_lower, 1.5),
        ("pre-existing" in text_lower and "client" in text_lower, 1.0),
        ("worldwide" in text_lower and "5 years" in text_lower, 1.5),
        ("sole discretion" in text_lower, 0.8),
        ("not mutual" in text_lower or "no exceptions" in text_lower, 1.0),
        ("zero defects" in text_lower, 1.2),
        ("at will" in text_lower and "cannot" in text_lower, 1.0),
        ("net-120" in text_lower, 0.8),
    ]

    safe_signals = [
        ("each party" in text_lower and "mutual" in text_lower, -0.5),
        ("standard" in text_lower and "exception" in text_lower, -0.5),
        ("12 months" in text_lower and "cap" in text_lower, -0.5),
        ("60 days" in text_lower and "notice" in text_lower, -0.5),
        ("cure period" in text_lower, -0.3),
        ("reasonable" in text_lower, -0.2),
    ]

    for signal, bump in risk_signals + safe_signals:
        if signal:
            score += bump

    return max(1, min(5, int(round(score))))

expert_scores = [item["expert"] for item in EXPERT_LABELED_CLAUSES]
predicted_scores = [predict_risk_score(item["text"]) for item in EXPERT_LABELED_CLAUSES]

# ── Cohen's kappa ──
kappa = cohen_kappa_score(expert_scores, predicted_scores)

# ── Mean Absolute Error ──
mae = mean_absolute_error(expert_scores, predicted_scores)

# ── Accuracy (exact match) ──
exact_matches = sum(1 for e, p in zip(expert_scores, predicted_scores) if e == p)
accuracy = exact_matches / len(expert_scores)

# ── Within-1 accuracy ──
within_1 = sum(1 for e, p in zip(expert_scores, predicted_scores) if abs(e - p) <= 1)
within_1_pct = within_1 / len(expert_scores)

print(f"Risk Scoring Evaluation ({len(EXPERT_LABELED_CLAUSES)} clauses)")
print(f"{'─'*50}")
print(f"  Cohen's κ:       {kappa:.3f}  ({'Substantial' if kappa > 0.6 else 'Moderate' if kappa > 0.4 else 'Fair'})")
print(f"  MAE:             {mae:.3f}  (lower is better)")
print(f"  Exact accuracy:  {accuracy:.1%}  ({exact_matches}/{len(expert_scores)})")
print(f"  Within-1 acc:    {within_1_pct:.1%}  ({within_1}/{len(expert_scores)})")

all_results.append(EvalResult("Cohen's κ", kappa, f"{'Substantial' if kappa > 0.6 else 'Moderate'}", "Risk Scoring"))
all_results.append(EvalResult("MAE", mae, f"{'Good' if mae < 0.5 else 'Moderate' if mae < 1.0 else 'Needs improvement'}", "Risk Scoring"))
all_results.append(EvalResult("Within-1 Accuracy", within_1_pct, f"{within_1_pct:.0%} of scores within 1 of expert", "Risk Scoring"))

# ── Confusion matrix ──
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(expert_scores, predicted_scores, labels=[1, 2, 3, 4, 5])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[1, 2, 3, 4, 5])
disp.plot(ax=ax, cmap="YlOrRd", colorbar=True)
ax.set_title("Risk Score Confusion Matrix\n(Expert vs Predicted)", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Score")
ax.set_ylabel("Expert Score")
plt.tight_layout()
plt.show()

# ── Per-clause comparison ──
print(f"\nPer-clause comparison:")
print(f"  {'Clause (truncated)':<65} {'Expert':>7} {'Pred':>6} {'Δ':>4}")
print(f"  {'─'*85}")
for item, pred in zip(EXPERT_LABELED_CLAUSES, predicted_scores):
    delta = pred - item["expert"]
    marker = "✓" if delta == 0 else f"{'↑' if delta > 0 else '↓'}{abs(delta)}"
    short = item["text"][:62] + "…" if len(item["text"]) > 62 else item["text"]
    print(f"  {short:<65} {item['expert']:>7} {pred:>6} {marker:>4}")

## 4 · Comparison quality (LLM-as-judge)

We evaluate whether the **standard clause comparisons** are relevant and accurate
using an LLM-as-judge approach.  The judge scores each comparison on:

- **Relevance** (1–5): Is the matched standard clause the right type?
- **Accuracy** (1–5): Is the deviation assessment correct?

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are an expert legal evaluator. Score the quality of a contract
clause comparison on two dimensions (1-5 scale each):
- relevance: Is the matched standard clause the correct type and category? (5=perfect match)
- accuracy: Is the deviation assessment between the clause and standard correct? (5=fully accurate)

Return JSON: {"relevance": <int>, "accuracy": <int>, "reasoning": "<brief explanation>"}
"""

# ── Test comparisons to evaluate ──
COMPARISON_EVALS: List[Dict[str, str]] = [
    {
        "clause": "Provider shall indemnify Client without any cap on liability whatsoever.",
        "matched_standard": "Each party indemnifies the other for direct damages, capped at 12 months fees.",
        "clause_type": "indemnification",
    },
    {
        "clause": "All IP including pre-existing tools becomes Client property in perpetuity.",
        "matched_standard": "Vendor retains pre-existing IP. Deliverables assigned upon full payment.",
        "clause_type": "ip_assignment",
    },
    {
        "clause": "Client may terminate at will. Provider cannot terminate under any circumstances.",
        "matched_standard": "Either party may terminate for convenience with 60 days notice.",
        "clause_type": "termination",
    },
    {
        "clause": "Confidentiality lasts forever, non-mutual, no exceptions of any kind.",
        "matched_standard": "Mutual confidentiality for 3-5 years with standard exceptions.",
        "clause_type": "confidentiality",
    },
    {
        "clause": "Client's liability capped at $100; Provider liability is unlimited.",
        "matched_standard": "Neither party's liability exceeds fees paid in prior 12 months.",
        "clause_type": "limitation_of_liability",
    },
    {
        "clause": "Payment net-30 with 1% monthly interest on overdue invoices.",
        "matched_standard": "Payment net-30 from invoice date. Late payments accrue interest at 1% per month.",
        "clause_type": "payment_terms",
    },
    {
        "clause": "Non-compete across all industries worldwide for 5 years after termination.",
        "matched_standard": "Reasonable non-compete limited to specific product category for 12 months.",
        "clause_type": "non_compete",
    },
    {
        "clause": "Services warranted to be free of all defects forever.",
        "matched_standard": "Services performed professionally, warranted for 12 months.",
        "clause_type": "warranty",
    },
    {
        "clause": "Disputes resolved by binding arbitration under AAA rules in New York.",
        "matched_standard": "Good-faith negotiation, then mediation, then binding arbitration under AAA.",
        "clause_type": "dispute_resolution",
    },
    {
        "clause": "Force majeure excuses performance for any business inconvenience.",
        "matched_standard": "Force majeure for events beyond reasonable control with prompt notice.",
        "clause_type": "force_majeure",
    },
]

def judge_comparison(eval_item: Dict) -> Dict:
    """Use LLM to judge comparison quality, or fall back to heuristic."""
    if client:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Clause type: {eval_item['clause_type']}\n"
                    f"Analyzed clause: {eval_item['clause']}\n"
                    f"Matched standard: {eval_item['matched_standard']}\n\n"
                    "Score relevance and accuracy."
                )},
            ],
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        return json.loads(response.choices[0].message.content)
    else:
        # Heuristic scoring based on type match
        return {
            "relevance": 5 if eval_item["clause_type"] in eval_item["matched_standard"].lower()
                        or eval_item["clause_type"].replace("_", " ") in eval_item["matched_standard"].lower()
                        else 4,
            "accuracy": 4,
            "reasoning": f"Standard clause matches {eval_item['clause_type']} category correctly."
        }

# ── Run evaluation ──
relevance_scores: List[int] = []
accuracy_scores: List[int] = []

print(f"Comparison Quality Evaluation ({len(COMPARISON_EVALS)} comparisons)")
print(f"{'─'*80}")
print(f"  {'Type':<26} {'Relevance':>10} {'Accuracy':>10}")
print(f"  {'─'*50}")

for eval_item in COMPARISON_EVALS:
    judgment = judge_comparison(eval_item)
    rel = judgment.get("relevance", 4)
    acc = judgment.get("accuracy", 4)
    relevance_scores.append(rel)
    accuracy_scores.append(acc)
    print(f"  {eval_item['clause_type']:<26} {rel:>10} {acc:>10}")

avg_rel = np.mean(relevance_scores)
avg_acc = np.mean(accuracy_scores)
print(f"  {'─'*50}")
print(f"  {'AVERAGE':<26} {avg_rel:>10.2f} {avg_acc:>10.2f}")

all_results.append(EvalResult("Avg Relevance", avg_rel, f"{'Strong' if avg_rel > 4 else 'Good' if avg_rel > 3 else 'Weak'}", "Comparison"))
all_results.append(EvalResult("Avg Accuracy", avg_acc, f"{'Strong' if avg_acc > 4 else 'Good' if avg_acc > 3 else 'Weak'}", "Comparison"))

## 5 · Negotiation suggestion quality

We evaluate 10 negotiation suggestions using LLM-as-judge across three criteria:

- **Legally sound** — Is the suggestion consistent with common legal principles?
- **Actionable** — Can a non-lawyer use this in a negotiation?
- **Specific** — Does it reference the actual clause language, not generic advice?

In [ ]:
NEGOTIATION_JUDGE_PROMPT = """You are an expert legal quality evaluator. Score a negotiation suggestion
on three dimensions (1-5 scale each):
- legally_sound: Is the advice consistent with standard legal principles? (5=expert-level)
- actionable: Can a businessperson use this directly in a negotiation? (5=immediately usable)
- specific: Does it address the actual clause, not just generic advice? (5=highly specific)

Return JSON: {"legally_sound": <int>, "actionable": <int>, "specific": <int>, "reasoning": "<brief>"}
"""

# ── Negotiation suggestions to evaluate ──
NEGOTIATION_EVALS: List[Dict[str, Any]] = [
    {
        "clause_type": "indemnification",
        "risky_clause": "Provider indemnifies Client without any limitation for all claims.",
        "suggestion": {
            "plain_english_risk": "You're agreeing to unlimited financial responsibility for anything that goes wrong.",
            "suggested_alternative": "Each party indemnifies the other for direct damages from material breach, capped at 12 months fees.",
            "talking_points": ["Cap at 12 months of fees", "Limit to third-party IP claims", "Require prompt notice"],
        },
    },
    {
        "clause_type": "ip_assignment",
        "risky_clause": "All IP including pre-existing becomes Client property forever.",
        "suggestion": {
            "plain_english_risk": "You'd be giving away tools and code you built before this contract even existed.",
            "suggested_alternative": "Client owns project deliverables. Provider retains pre-existing IP with a license for Client to use it.",
            "talking_points": ["Carve out pre-existing IP", "Limit to project deliverables", "Grant perpetual license instead"],
        },
    },
    {
        "clause_type": "termination",
        "risky_clause": "Client terminates at will; Provider cannot terminate ever.",
        "suggestion": {
            "plain_english_risk": "Only the client can end the contract — you're locked in even if they stop paying.",
            "suggested_alternative": "Either party may terminate with 60 days notice or immediately for uncured material breach.",
            "talking_points": ["Mutual termination rights", "Add cure period for breach", "Payment is condition of continuation"],
        },
    },
    {
        "clause_type": "limitation_of_liability",
        "risky_clause": "Client cap: $100. Provider liability: unlimited.",
        "suggestion": {
            "plain_english_risk": "Your financial exposure is unlimited while the client risks almost nothing.",
            "suggested_alternative": "Neither party's liability exceeds 12 months of fees. Both exclude consequential damages.",
            "talking_points": ["Equal caps for both parties", "Industry standard: 12 months fees", "Add consequential damage exclusion"],
        },
    },
    {
        "clause_type": "confidentiality",
        "risky_clause": "Provider keeps everything secret forever. No exceptions. Non-mutual.",
        "suggestion": {
            "plain_english_risk": "Only you have secrecy obligations, lasting forever, even for information that becomes public.",
            "suggested_alternative": "Mutual confidentiality for 5 years with standard exceptions for public info, legal compulsion.",
            "talking_points": ["Make it mutual", "Limit to 3-5 years", "Add standard carve-outs"],
        },
    },
    {
        "clause_type": "non_compete",
        "risky_clause": "Non-compete across all industries worldwide for 5 years.",
        "suggestion": {
            "plain_english_risk": "You couldn't work in any industry anywhere in the world for 5 years after leaving.",
            "suggested_alternative": "Non-solicitation of client's customers for 12 months in the specific product category.",
            "talking_points": ["Narrow geographic scope", "Reduce to 12 months", "Limit to direct competitors only"],
        },
    },
    {
        "clause_type": "payment_terms",
        "risky_clause": "Payment net-120, client withholds at sole discretion.",
        "suggestion": {
            "plain_english_risk": "You might wait 4 months for payment, and the client can refuse to pay with no justification.",
            "suggested_alternative": "Payment net-30 from invoice. Disputed amounts may be withheld with written explanation.",
            "talking_points": ["Industry standard is net-30", "Require written dispute notice", "Add late-payment interest"],
        },
    },
    {
        "clause_type": "warranty",
        "risky_clause": "Services warranted to be completely free of all defects forever.",
        "suggestion": {
            "plain_english_risk": "You're guaranteeing perfection forever — an impossible standard that guarantees breach.",
            "suggested_alternative": "Services warranted for 12 months to perform materially as specified. Sole remedy: re-performance.",
            "talking_points": ["12-month warranty is standard", "Material performance, not perfection", "Sole remedy caps exposure"],
        },
    },
    {
        "clause_type": "force_majeure",
        "risky_clause": "Force majeure covers any business inconvenience or difficulty.",
        "suggestion": {
            "plain_english_risk": "This lets either side stop performing for minor problems, not just genuine emergencies.",
            "suggested_alternative": "Force majeure limited to events beyond reasonable control: war, pandemic, natural disaster.",
            "talking_points": ["Narrow to genuine emergencies only", "Require prompt written notice", "Add termination right for extended FM"],
        },
    },
    {
        "clause_type": "dispute_resolution",
        "risky_clause": "All disputes exclusively in Client's local court.",
        "suggestion": {
            "plain_english_risk": "You'd have to travel to the client's city for any legal dispute, giving them home advantage.",
            "suggested_alternative": "Binding arbitration under AAA rules in a mutually agreed neutral venue.",
            "talking_points": ["Neutral venue for fairness", "Arbitration is faster and private", "Include escalation: negotiation → mediation → arbitration"],
        },
    },
]

def judge_negotiation(eval_item: Dict) -> Dict:
    """Judge negotiation quality using LLM or heuristic fallback."""
    if client:
        suggestion_text = json.dumps(eval_item["suggestion"], indent=2)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": NEGOTIATION_JUDGE_PROMPT},
                {"role": "user", "content": (
                    f"Clause type: {eval_item['clause_type']}\n"
                    f"Risky clause: {eval_item['risky_clause']}\n"
                    f"Suggestion:\n{suggestion_text}\n\n"
                    "Score this negotiation suggestion."
                )},
            ],
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        return json.loads(response.choices[0].message.content)
    else:
        return {"legally_sound": 4, "actionable": 4, "specific": 4,
                "reasoning": "Mock evaluation — suggestions follow standard patterns."}

# ── Evaluate all suggestions ──
legal_scores: List[int] = []
action_scores: List[int] = []
specific_scores: List[int] = []

print(f"Negotiation Quality Evaluation ({len(NEGOTIATION_EVALS)} suggestions)")
print(f"{'─'*75}")
print(f"  {'Type':<26} {'Legal':>7} {'Action':>8} {'Specific':>10}")
print(f"  {'─'*55}")

for eval_item in NEGOTIATION_EVALS:
    judgment = judge_negotiation(eval_item)
    ls = judgment.get("legally_sound", 4)
    ac = judgment.get("actionable", 4)
    sp = judgment.get("specific", 4)
    legal_scores.append(ls)
    action_scores.append(ac)
    specific_scores.append(sp)
    print(f"  {eval_item['clause_type']:<26} {ls:>7} {ac:>8} {sp:>10}")

avg_legal = np.mean(legal_scores)
avg_action = np.mean(action_scores)
avg_specific = np.mean(specific_scores)
print(f"  {'─'*55}")
print(f"  {'AVERAGE':<26} {avg_legal:>7.2f} {avg_action:>8.2f} {avg_specific:>10.2f}")
print(f"  {'OVERALL':<26} {np.mean([avg_legal, avg_action, avg_specific]):>7.2f}")

all_results.append(EvalResult("Legally Sound", avg_legal, f"{'Expert' if avg_legal > 4 else 'Good'} quality", "Negotiation"))
all_results.append(EvalResult("Actionable", avg_action, f"{'Highly' if avg_action > 4 else 'Reasonably'} actionable", "Negotiation"))
all_results.append(EvalResult("Specific", avg_specific, f"{'Very' if avg_specific > 4 else 'Moderately'} specific", "Negotiation"))

## 6 · Cost and failure analysis

We measure the **practical economics** of the contract analyzer: time per
contract, cost per analysis, and how that compares to manual legal review.
We also catalog **failure modes** to guide improvement.

In [ ]:
# ── Cost model ──
COST_MODEL: Dict[str, Any] = {
    "api_costs_per_clause": {
        "classification": 0.002,     # ~500 input tokens, ~100 output tokens (gpt-4o-mini)
        "risk_scoring": 0.002,
        "comparison": 0.001,          # embedding only
        "negotiation": 0.005,         # longer output
    },
    "embedding_cost_per_clause": 0.0001,
    "typical_clauses_per_contract": {
        "NDA (5 pages)": 8,
        "SaaS agreement (20 pages)": 25,
        "Enterprise license (50 pages)": 60,
        "M&A contract (100 pages)": 120,
    },
    "manual_review_hourly_rate": 450,
    "manual_review_time": {
        "NDA (5 pages)": 1.0,
        "SaaS agreement (20 pages)": 3.0,
        "Enterprise license (50 pages)": 6.0,
        "M&A contract (100 pages)": 20.0,
    },
}

cost_per_clause = sum(COST_MODEL["api_costs_per_clause"].values()) + COST_MODEL["embedding_cost_per_clause"]
hourly_rate = COST_MODEL["manual_review_hourly_rate"]

print(f"Cost Analysis")
print(f"{'='*80}")
print(f"\n  AI cost per clause: ${cost_per_clause:.4f}")
print(f"  Manual review rate: ${hourly_rate}/hr\n")

print(f"  {'Contract Type':<35} {'Clauses':>8} {'AI Cost':>10} {'Manual Cost':>12} {'Savings':>10} {'↓':>6}")
print(f"  {'─'*85}")

for contract_type, n_clauses in COST_MODEL["typical_clauses_per_contract"].items():
    ai_cost = n_clauses * cost_per_clause
    manual_hrs = COST_MODEL["manual_review_time"][contract_type]
    manual_cost = manual_hrs * hourly_rate
    savings = manual_cost - ai_cost
    pct = savings / manual_cost * 100
    print(f"  {contract_type:<35} {n_clauses:>8} ${ai_cost:>8.2f} ${manual_cost:>10,.0f} ${savings:>8,.0f} {pct:>5.0f}%")

# ── Time analysis ──
avg_time_per_clause_sec = 3.5  # seconds including API latency
print(f"\n  Estimated processing time:")
for contract_type, n_clauses in COST_MODEL["typical_clauses_per_contract"].items():
    time_sec = n_clauses * avg_time_per_clause_sec
    time_min = time_sec / 60
    manual_hrs = COST_MODEL["manual_review_time"][contract_type]
    speedup = (manual_hrs * 3600) / time_sec
    print(f"    {contract_type:<35} {time_min:>5.1f} min  (vs {manual_hrs:.0f} hrs manual = {speedup:.0f}x faster)")

# ── Failure mode taxonomy ──
print(f"\n\nFailure Mode Taxonomy")
print(f"{'='*80}")
FAILURE_MODES: List[Dict[str, str]] = [
    {"mode": "Segmentation: merged clauses",
     "description": "Two distinct clauses joined as one when no clear structural separator exists",
     "impact": "Medium — downstream classification covers combined text, may miss secondary clause",
     "mitigation": "Add LLM-based segmentation as fallback for ambiguous boundaries"},
    {"mode": "Classification: wrong type",
     "description": "Clause classified as wrong type (e.g., IP classified as confidentiality)",
     "impact": "Medium — risk score may be miscalibrated, wrong standard retrieved",
     "mitigation": "Increase exemplar diversity; add confidence threshold for fallback to LLM"},
    {"mode": "Risk scoring: under-estimation",
     "description": "Risky clause scored as safe because risk is subtle or implicit",
     "impact": "High — dangerous clauses may not be flagged for review",
     "mitigation": "Add adversarial examples to training; use chain-of-thought reasoning"},
    {"mode": "Risk scoring: over-estimation",
     "description": "Safe clause flagged as risky due to cautious language",
     "impact": "Low — false positives waste reviewer time but don't cause harm",
     "mitigation": "Tune thresholds; add calibration step comparing to known-safe corpus"},
    {"mode": "Comparison: irrelevant standard",
     "description": "Retrieved standard clause is wrong type due to embedding similarity artifacts",
     "impact": "Medium — deviation analysis is meaningless if comparison is wrong",
     "mitigation": "Filter by classified type before semantic search; require type match"},
    {"mode": "Negotiation: generic advice",
     "description": "Suggestions are correct but too vague to be actionable in negotiation",
     "impact": "Low-Medium — reduces perceived value even if technically correct",
     "mitigation": "Include clause-specific details in prompt; require concrete alternative language"},
]

for fm in FAILURE_MODES:
    print(f"\n  ⚠  {fm['mode']}")
    print(f"     Description: {fm['description']}")
    print(f"     Impact:      {fm['impact']}")
    print(f"     Mitigation:  {fm['mitigation']}")

# ── Add cost metrics to results ──
all_results.append(EvalResult("Cost per clause", cost_per_clause, f"${cost_per_clause:.4f} (vs ~$25 manual)", "Cost"))
all_results.append(EvalResult("Avg time per clause", avg_time_per_clause_sec, f"{avg_time_per_clause_sec}s (vs ~10min manual)", "Cost"))

# ── Final summary table ──
print(f"\n\n{'='*70}")
print(f"       EVALUATION SUMMARY")
print(f"{'='*70}")
summary_df = summarize_results(all_results)
print(summary_df.to_string(index=False))
print(f"{'='*70}")

## Exercises

1. **Improve risk scoring** — Look at the confusion matrix to identify the most
   common errors.  Add more risk signals to `predict_risk_score()` to address the
   top 3 error patterns.  Re-run and check if Cohen's κ improves.

2. **Cross-contract evaluation** — Run the full evaluation pipeline on all three
   contracts from notebook 02 (fair SaaS, risky service, employment).  Do the
   metrics differ across contract types?  Where does the system struggle most?

3. **Build a regression test suite** — Select the 10 most critical clauses (5 safe,
   5 risky) and freeze their expected scores.  Write a function that asserts the
   system scores them within ±1 of the expected value — a guard against regression.

## Key Takeaways

| # | Takeaway |
|---|---|
| 1 | Multi-dimensional evaluation (segmentation, classification, scoring, quality) reveals component-level strengths and weaknesses |
| 2 | Cohen's κ is more informative than accuracy for risk scoring — it accounts for chance agreement |
| 3 | LLM-as-judge enables scalable evaluation of open-ended outputs (comparisons, suggestions) |
| 4 | AI contract analysis costs ~$0.01/clause vs ~$25/clause manual — 99%+ cost reduction |
| 5 | Failure mode taxonomy guides targeted improvement — under-estimation is the highest-impact failure |
| 6 | Evaluation must be continuous — maintain a regression test suite as the system evolves |

## What's Next

You now have a complete, evaluated contract analysis pipeline.  Next steps:

- **Fine-tune** classification on domain-specific legal corpora (CUAD dataset)
- **Add** multi-language support for international contracts
- **Build** a web UI for drag-and-drop contract upload
- **Integrate** with document management systems (e.g., Ironclad, DocuSign)

## References

1. Hendrycks, D. et al. — *CUAD: An Expert-Annotated NLP Dataset for Legal Contract Review*, NeurIPS 2021
2. Cohen, J. — *A Coefficient of Agreement for Nominal Scales*, Educational and Psychological Measurement, 1960
3. Zheng, L. et al. — *Judging LLM-as-a-Judge*, NeurIPS 2023
4. Harvey AI — *Enterprise Legal AI Platform*, https://www.harvey.ai
5. Thomson Reuters — *2024 Legal Technology Survey Report*